# BlisterTrace / EfficientAD-S / VisA capsules

**Exactly one full model:** `visa_capsules`, 70,000 optimizer steps, batch 1,
256 x 256 RGB, FP32 (`amp=False`), genuine pretrained PDN teacher,
trainable student + feature autoencoder. This is EfficientAD, not EfficientNet.
Normal-only training; disjoint normal validation and calibration; held-out
normal/defective images and real pixel masks for evaluation. No random-teacher
or synthetic-data training substitute. Dataset terms: **CC BY 4.0**.

## Read before starting
1. Import this notebook into its own Kaggle notebook. Select **GPU T4** and
   enable Internet. No API key or manual dataset upload is needed.
2. Run Steps 1-5. Step 5 pauses after 10 actual training steps for a mandatory
   backup download. After downloading that ZIP, confirm in Step 6 and run it
   to continue the same 70,000-step schedule, evaluate and package. Run All
   alone pauses at the safety gate; it does not finish the whole model.
3. Actual model/checkpoint/source files are under
   `/kaggle/working/blistertrace_efficientad_visa_capsules`. Dataset archives/images
   are disposable scratch under `/kaggle/temp`, excluded from model ZIPs.
   The notebook asserts the output path; it never chooses `/content`.
4. Checkpoints: first at step **10** (part of the full run), then every **500**
   steps; normal validation every 1,000. Last and previous checkpoints have
   integrity receipts. An uncompressed rolling Recovery ZIP is written at
   step 10 and every 5,000 steps; calibrated Weights ZIP immediately on completion.
5. **Download the first Recovery ZIP while training is paused**, and confirm
   you have the actual file before continuing. Save newer snapshots during
   the long run when practical. Refresh Output and use the file menu; a code
   cell queued behind training cannot download/save immediately.
6. Save a Kaggle version with **Quick Save + save output files**, when offered,
   after verifying the actual checkpoints appear in Output. **Save & Run All
   starts a fresh session**; it is not the way to preserve an ongoing run.
7. Download the final Submission ZIP, not only the `.ipynb`. Copying a notebook
   to another account/computer does not copy its model files.

**No notebook can guarantee survival of a deleted cloud VM.** `/kaggle/working`
makes files visible/saveable, but files still need a successful external download,
saved output version, or verified external copy. The notebook cannot click Save
Version or approve a browser download for you. Quotas and session availability
remain Kaggle-controlled.
This workflow does not claim a winning model, camera accuracy, or Hailo conversion.


## Step 1 - Environment and minimal dependencies
Keep the provider's matching Torch/torchvision pair. Setup and file rescue
can run without GPU quota; only the training cell requires a GPU.


In [ ]:
import os,sys,json,subprocess,platform,shutil
from pathlib import Path
assert Path('/kaggle/working').is_dir(), 'This notebook is for Kaggle.'
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'PyYAML>=6,<7','scipy>=1.15,<2','scikit-learn>=1.6,<2',
    'opencv-python-headless==4.11.0.86','psutil>=6,<8'])
import torch,torchvision,numpy,yaml
print('Python:',platform.python_version(),'Torch:',torch.__version__,
      'Torchvision:',torchvision.__version__)
print('CUDA:',torch.cuda.is_available(),'GPUs:',torch.cuda.device_count())
if torch.cuda.is_available(): print('GPU 0:',torch.cuda.get_device_name(0))


## Step 2 - Fixed category, output path and optional checkpoint restore
Leave START_MODE=`fresh` for a genuinely new model. It will still reuse a
matching live checkpoint if you rerun training in the same session.
For a new session with saved weights: attach your own downloaded Recovery,
Weights or Submission ZIP as a private Kaggle input, set START_MODE=`resume`,
and set RESUME_ARCHIVE to its full `/kaggle/input/...zip` path.
Resume without a real matching checkpoint fails instead of silently retraining.

EXTERNAL_BACKUP_DIR is optional: set it only to an already mounted storage
directory you control. Its durability depends on that storage service.
Leave it empty if no such mount exists. Downloads/saved Kaggle versions
are still required when it is empty. Never put credentials in this notebook.


In [ ]:
RUN_NAME='visa_capsules'
FULL_STEPS=70000
START_MODE='fresh'  # 'resume' requires actual checkpoint files
RESUME_ARCHIVE=''  # own trusted ZIP under /kaggle/input/...; NOT an .ipynb
EXTERNAL_BACKUP_DIR=''  # optional existing mounted storage directory
MAX_TRAIN_HOURS=8  # safe-pause budget; does not extend Kaggle session limits
WORK=Path('/kaggle/working')/('blistertrace_efficientad_'+RUN_NAME)
SCRATCH=Path('/kaggle/temp')/('blistertrace_efficientad_'+RUN_NAME)
OUTPUT=WORK/'runs'/RUN_NAME
for p in (WORK,SCRATCH,OUTPUT,WORK/'exports'):p.mkdir(parents=True,exist_ok=True)
assert WORK.resolve().is_relative_to(Path('/kaggle/working').resolve())
assert START_MODE in {'fresh','resume'}
if RESUME_ARCHIVE: assert START_MODE=='resume', 'Set START_MODE=resume for restoration.'
if EXTERNAL_BACKUP_DIR: assert Path(EXTERNAL_BACKUP_DIR).is_dir(), 'Mount external storage first.'
os.environ.update(BT_WORK=str(WORK),BT_RUN_NAME=RUN_NAME,
    BT_MAX_TRAIN_SECONDS=str(MAX_TRAIN_HOURS*3600),
    BT_EXTERNAL_BACKUP_DIR=EXTERNAL_BACKUP_DIR)
os.chdir('/kaggle/working')
print('ONE CATEGORY:',RUN_NAME,'| FULL STEPS:',FULL_STEPS,'| FP32, batch 1')
print('VISIBLE CHECKPOINT:',OUTPUT/'last.pt')
print('DOWNLOAD FOLDER:',WORK/'exports')
print('External copy:',EXTERNAL_BACKUP_DIR or 'NOT CONFIGURED: download/save Output files yourself')


## Step 3 - Verify and unpack the embedded source; restore if requested
Only code/licenses are embedded. Model and data downloads happen next.
Restored code is not executed: this notebook uses its own pinned source.
Archives must match the category and source version and pass checksums.
Load only trusted checkpoints produced by this workflow. This is not an
automatic importer for older multi-category notebook checkpoints.


In [ ]:
SOURCE_B64='...'
SOURCE_SHA256='f45364b8335d0623c6d1dbabf0ac3341474495bf901b09920001777140c81331'
import ast,base64,hashlib,io,zipfile
import psutil
lock=WORK/'trainer.lock'
if lock.exists():
    owner=int(lock.read_text())
    assert not psutil.pid_exists(owner), 'A live trainer owns this workspace. Do not rerun setup while training.'
raw=base64.b64decode(SOURCE_B64)
assert hashlib.sha256(raw).hexdigest()==SOURCE_SHA256
for module in list(sys.modules):
    if module=='single_runner' or module=='efficientad' or module.startswith('efficientad.'):
        del sys.modules[module]
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        target=(WORK/name).resolve()
        assert target.is_relative_to(WORK.resolve())
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(archive.read(name))
PACKAGE=WORK/'models/02_efficientad'
for folder in [str(WORK),str(PACKAGE)]:
    if folder in sys.path:sys.path.remove(folder)
    sys.path.insert(0,folder)
from efficientad.checkpoint_safety import atomic_json,restore_archive,choose_checkpoint
atomic_json(WORK/'payload_identity.json',{'sha256':SOURCE_SHA256,'run_name':RUN_NAME})
if RESUME_ARCHIVE:
    archive=Path(RESUME_ARCHIVE)
    assert archive.is_file(), 'Requested checkpoint ZIP does not exist.'
    print('Restored files:',restore_archive(archive,WORK,RUN_NAME))
if START_MODE=='resume':
    assert (OUTPUT/'last.pt').is_file() or (OUTPUT/'previous.pt').is_file() or (OUTPUT/'best.pt').is_file(),         'Resume requested but no weights exist. Notebook logs cannot restore weights.'
from efficientad.utils import environment
atomic_json(WORK/'environment.json',environment())
(WORK/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Pinned source verified. Output stays under /kaggle/working.')


## Step 4 - Download VisA capsules, the real teacher and ImageNette penalty
Only this selected category is prepared. VisA downloads its official ~1.93 GB
archive; MVTec downloads the public ~5.27 GB mirror but extracts one category.
ImageNette adds about 99 MB. These are source sizes from the checked snapshot.
Teacher and VisA publisher checksums are checked; MVTec mirror gets a local
SHA256 receipt without an independently verified publisher archive checksum.
Data overlap/mask integrity checks run before fitting. No API key is needed.
Scratch is replaceable after a VM reset; regeneration must match the recorded
data fingerprints before resuming. A changed source is rejected.


In [ ]:
from single_runner import run_process,prepare_evidence
from cloud_data import prepare_runs,prepare_penalty
archive_name='VisA_20220922.tar' if RUN_NAME=='visa_capsules' else 'mvtec_ad_kaggle.zip'
cached=(SCRATCH/'downloads'/archive_name).is_file()
required_gib=2 if cached else (5 if RUN_NAME=='visa_capsules' else 9)
assert shutil.disk_usage(SCRATCH).free > required_gib*1024**3,             f'Less than {required_gib} GiB free: inspect storage before downloading. Do not delete model checkpoints.'
run_process([PACKAGE/'download_weights.py','--output',WORK/'artifacts/teacher'],
            WORK,WORK/'teacher_download.log')
PREPARED,UNAVAILABLE=prepare_runs(SCRATCH,[RUN_NAME],strict=True)
assert set(PREPARED)=={RUN_NAME}, 'Only one ready category is permitted.'
PENALTY_ROOT=prepare_penalty(SCRATCH)
prepare_evidence(WORK,SCRATCH,PREPARED,RUN_NAME)
print('READY:',RUN_NAME,'| Data provenance saved in',WORK/'evidence')


## Step 5 - First real checkpoint and mandatory download gate
This performs only the first 10 steps of the original 70,000-step schedule,
saves optimizer/model/RNG state, reads the checkpoint back with PyTorch and
creates the actual Recovery ZIP. This is not a separate short-horizon model,
timing trial, synthetic run or reset: Step 6 continues the same saved training.
No hours-long training starts before you can download a real checkpoint.

After this cell finishes, refresh Output, expand the workspace's `exports`
folder, download the Recovery ZIP and verify it exists on your computer.
Then confirm below in Step 6. If a matching run already has a checkpoint,
this cell rescues it instead of resetting it.


In [ ]:
assert torch.cuda.is_available(), 'Training needs a GPU. Existing files can still be rescued without one.'
assert FULL_STEPS==70000
from single_runner import train_one
from IPython.display import FileLink,display
TRAIN_RESULT=train_one(WORK,SCRATCH,RUN_NAME,PREPARED,PENALTY_ROOT,
    mode=START_MODE,initial_backup_gate=True)
print('INITIAL CHECKPOINT RESULT:',TRAIN_RESULT,flush=True)
for path in sorted((WORK/'exports').glob('*.zip')):
    display(FileLink(str(path.relative_to('/kaggle/working'))))
print('DOWNLOAD THE ACTUAL RECOVERY ZIP, then confirm and run Step 6.',flush=True)


## Step 6 - Confirm the backup, continue all 70,000 steps, evaluate and package
Change INITIAL_BACKUP_CONFIRMED to True **only after** the Recovery ZIP has
actually downloaded and opens on your computer. A link or notebook download
is not a saved checkpoint. This is a manual confirmation; this runtime cannot
inspect your computer's Downloads folder. An attached restored archive is also
external storage, but confirm that file before continuing.

This cell resumes saved state and performs the remaining training, calibration,
evaluation and final bundle. It never starts another category. Subsequent
checkpoints are every 500 steps; rolling recovery ZIPs every 5,000. A completed
calibrated Weights ZIP is saved **before** evaluation so an evaluation failure
does not require retraining. Rerun this cell after a recoverable interruption.

SIGINT/SIGTERM request a safe checkpoint after a completed step; the parent
waits for its owned subprocess to finish. Hard VM deletion/SIGKILL cannot be
caught. An 8-hour application budget requests a safe pause; it cannot override
an earlier Kaggle quota/session termination. A paused run is labeled PAUSED,
not complete. Keep paths, schedule, resolution, category, FP32 and batch size unchanged.


In [ ]:
INITIAL_BACKUP_CONFIRMED=False  # set True after the actual ZIP is safely downloaded
from single_runner import train_one,evaluate_one
from IPython.display import FileLink,display
if not INITIAL_BACKUP_CONFIRMED:
    print('SAFETY PAUSE: download the Recovery ZIP from Output, then set INITIAL_BACKUP_CONFIRMED=True and rerun this cell.',flush=True)
else:
    assert torch.cuda.is_available(), 'Enable a GPU to continue the remaining training.'
    TRAIN_RESULT=train_one(WORK,SCRATCH,RUN_NAME,PREPARED,PENALTY_ROOT,mode='resume')
    print('TRAINING RESULT:',TRAIN_RESULT,flush=True)
    if TRAIN_RESULT['status']=='complete':
        BUNDLE,SUMMARY=evaluate_one(WORK,RUN_NAME)
        print(json.dumps(SUMMARY,indent=2))
        print('FINAL MODEL + EVIDENCE:',BUNDLE,flush=True)
        display(FileLink(str(BUNDLE.relative_to('/kaggle/working'))))
    else:
        print('PAUSED: download the Recovery ZIP and rerun this cell to resume.',flush=True)


## Step 7 - Inspect/download saved artifacts (after training ends)
This cell lists real files with their save times and verifies checkpoints.
It does not train, copy datasets or spend time compressing a full workspace.
During training, use the right-hand Output file browser instead; another
notebook cell queues behind the active one.


In [ ]:
from datetime import datetime,timezone
from efficientad.checkpoint_safety import verify_receipt
from IPython.display import FileLink,display
for name in ['last.pt','previous.pt','best.pt']:
    path=OUTPUT/name
    if path.is_file():
        print(name,verify_receipt(path),flush=True)
for path in sorted((WORK/'exports').glob('*.zip')):
    print(path,'MiB:',round(path.stat().st_size/1024**2,1),
          'modified UTC:',datetime.fromtimestamp(path.stat().st_mtime,timezone.utc).isoformat())
    display(FileLink(str(path.relative_to('/kaggle/working'))))
print('Verify the ZIP actually appears in your computer Downloads before ending the session.')


## Recovery and what the files mean
- `last.pt`: optimizer, model, RNG/sampler, step and best-selection state for resume.
- `previous.pt`: previous verified checkpoint; used explicitly if latest bytes fail validation.
- `best.pt`: best normal-validation weights, finalized map normalization and calibrated
  thresholds after the full schedule; not an optimizer resume checkpoint.
- `*_Recovery.zip`: resumable weights, config, source, data evidence and hashes; no raw data.
- `*_Weights.zip`: completed calibrated weights saved before held-out evaluation.
- `*_Submission.zip`: weights plus held-out metrics, heatmaps, contracts and provenance.

**Same live session:** rerun Step 6 with backup confirmation; it reports the saved step and continues.
**Fresh session:** upload your own saved ZIP as private input, import this exact
notebook version, set START_MODE=`resume` and RESUME_ARCHIVE, then run Steps 1-6.
Data is regenerated from the same source, and source/config/data identities must match.
Missing files or changed identities stop recovery rather than secretly starting over.
These files do not reconstruct another user's deleted VM or missing `.pt` weights.

To stop deliberately, Cancel Run once and allow the checkpoint/child process to finish.
Avoid repeated interrupts. For a quota/platform error, first rescue/download existing
Output files; do not reset/delete the session hoping to restore its weights.

## Scope and references
This category benchmark does not establish blister-camera/SKU accuracy, unknown-defect
performance on arbitrary products, or Hailo compatibility. Scores are not probabilities
or severity labels. A new camera needs normal reference data and independent labeled
evaluation. Preserve MVTec noncommercial/share-alike terms where applicable.
Optional anomaly inference will be integrated only after actual trained artifacts exist.
Llama/ExecuTorch remains independent of this vision/anomaly training.

The Arm course motivates explicit model contracts, quality checks and target measurement;
EfficientAD is a project extension, not a model taught by the course.
- [EfficientAD paper](https://arxiv.org/abs/2303.14535)
- [VisA owner and license](https://github.com/amazon-science/spot-diff)
- [MVTec dataset and terms](https://www.mvtec.com/research-teaching/datasets/mvtec-ad)
- [MVTec mirror](https://www.kaggle.com/datasets/ipythonx/mvtec-ad)
- [Kaggle notebooks/save behavior](https://www.kaggle.com/docs/notebooks)
- [Kaggle advanced save options](https://www.kaggle.com/discussions/product-feedback/161328)
- [Arm course](https://learn.arm.com/courses/executorch-on-arm/)
